In [1]:
import pandas as pd
import numpy as np

def query_for_skills():
    try:
        import snowflake.connector
    except:
        !pip install snowflake-connector-python
        import snowflake.connector


    import base64
    WAREHOUSE = 'COMPUTE_WH'
    USER = 'OMAR_CLAFLIN'
    code = b'T2xsaWUyMSE='
    PASSWORD = base64.b64decode(code).decode("utf-8")
    ACCOUNT='[REDACTED]'
    DATABASE = '[REDACTED]'

    ctx = snowflake.connector.connect(
        user=USER,
        password=PASSWORD,
        account=ACCOUNT, 
        database=DATABASE
    #     schema=SCHEMA
        )

    # Retrieving a Snowflake Query ID
    cur = ctx.cursor()
    cur.execute("USE warehouse COMPUTE_WH;")
    cur.execute("USE DATABASE [REDACTED];")
    cur.execute("SELECT DISTINCT RL_TOP_LEVEL_SKILL_ID from [REDACTED].views.math_answers_fact")
    print(cur.sfqid)

    # Get the results from a query.
    query_id=cur.sfqid
    cur.get_results_from_sfqid(query_id)
    results = cur.fetchall()
    cur.close()
    #print(f'{results}')
    return results




#probably need to clean out s3 location

def downloadFromS3(bucket_name = '[REDACTED]', local_dir = 'AdaptiveMath'):
    #move from s3
    from boto3.session import Session
    import os

    if local_dir in os.listdir('.'):
        import shutil
        shutil.rmtree(local_dir)
        print(local_dir, ' removed.')
    os.mkdir(local_dir)

    session = Session()
    s3 = session.resource('s3')
    # s3_connection=boto.connect_s3()
    # s3_client = boto3.client('s3')
    bucket = s3.Bucket(bucket_name)
    # for file in os.listdir('AdaptiveMath'):
    # bucket=s3.Bucket(bucket_name)
    for s3_file in bucket.objects.all():
        print(s3_file)
        file_object = s3_file.key
        filename=str(file_object.split('/')[-1])
        bucket.download_file(file_object,local_dir+'/'+filename)
        

def read_filter_csv(filename, names, skill_id):
    df = pd.read_csv(filename, names =names)
    df = df[df['rl_top_level_skill_id']== skill_id][['student_id','math_question_id','correctness','created_at']]
    return df

def loadAndFilterIntoDataframe(skill_id = '02a3bfaa479de311b77c005056801da1', local_dir='AdaptiveMathData', 
                              parallel = True):
    #load and extract gzip csv files
    #need to dump into a user x question matrix (for each skill probably separately)
    import pandas as pd
    import os

    #header info so the file gets read correctly
#     names =['student_id', 'session_id', 'created_at', 'product', 'math_question_id',
#                                      'rl_top_level_skill_id', 'duration_seconds', 'correctness', 'attempts',
#                                     'answer']
    names =['student_id', 'session_id', 'created_at', 'math_question_id',
                                    'correctness', 'rl_top_level_skill_id']


    if parallel:
        files = os.listdir(local_dir)
        filelist = [file for file in files if file[-6:] == 'csv.gz']
        
        #test only a few
        #filelist=filelist[:10]
        from joblib import Parallel, delayed
        results = Parallel(n_jobs=-2)(delayed(read_filter_csv)(local_dir + '/'+i, 
                                names, skill_id) for i in filelist)
        df = pd.concat(results,axis=0,ignore_index=True)        
    if not parallel:
        #non parallel way to do it
        big_df_list=[]
        for file in os.listdir(local_dir)[:]:
            if file[-6:] == 'csv.gz':
                #print('file: ',file)
                df = pd.read_csv(local_dir + '/'+file,
                                 names =names)
                #print(df)
            #print(skill_id)
            df = df[df['rl_top_level_skill_id']== skill_id][['student_id','math_question_id','correctness','created_at']]
            #print(df)
            big_df_list.append(df)

        df = pd.concat(big_df_list,axis=0,ignore_index=True)
        
    import time
    start = time.time()
    df = df.sort_values(by=['created_at'])
    print('done sorting, took ', time.time()-start, ' to sort.')
    
    return df[['student_id','math_question_id','correctness']]

def returnTable(df):
    #df into table
    table = pd.pivot_table(df, values='correctness', index='student_id', columns='math_question_id',aggfunc='first')
    table = table/100
    import numpy as np
    table = table.apply(np.floor)    #they put 50, 25 as correctness sometimes for 2nd,3rd attmepted correct
    # table['general_score']=table.mean(axis=1)
    # table['num_weights']=table.notna().sum(axis=1)
    return table        

def three_param_logistic(theta,a,b,c):
    #theta: student ability
    #b: difficulty parameter
    #a: discrimination parameter
    #c: guessing parameter
    #d: inattention parameter
    import numpy as np
    return c + (1-c)*np.exp(a*(theta-b))/(1+np.exp(a*(theta-b)))

def four_param_logistic(theta,a,b,c,d):
    #theta: student ability
    #b: difficulty parameter
    #a: discrimination parameter
    #c: guessing parameter
    #d: inattention parameter
    import numpy as np
    return c + (d-c)*np.exp(a*(theta-b))/(1+np.exp(a*(theta-b)))



def estimate_parameters_for_skill(table, thetas, PLOT_ON=True, FOUR_PL=True,
                                 bounds = None):
    #table: table which is all users X all items for that skill, pivoted from pandas
    #needs to include a 'general_score', and a 'num_weights'
    MINIMUM_DATA_POINTS = 30
    #FOUR_PL=False
    from scipy.optimize import curve_fit
    import numpy as np
    
    num_items = len(table.columns)
    
    all_discriminability = np.empty((num_items))
    all_difficulty = np.empty((num_items))
    all_guessing = np.empty((num_items))
    all_discriminability[:] = np.nan
    all_difficulty[:] = np.nan
    all_guessing[:] = np.nan
    model = three_param_logistic
    if bounds == None:
        bounds = ((1,0,0),(100,1,.5)) #fix alpha as positive only
    #p0 = [.5, .5, 0]
    
    if FOUR_PL:
        model = four_param_logistic
        all_attention_errors = np.empty((num_items))
        all_attention_errors[:] = np.nan
        if bounds == None:
            bounds = ((1,0,0,.5),(100,1,.5,1))
        #p0 = [.5, .5, 0, 1]


    #LATER CAN PARALLELIZE THIS TO ESTIMATE ALL ITEMS AT ONCE
    for item_num in range(num_items):
        item=table.columns[item_num]
        item_series = table[[item]][table[item].notna()]
        item_thetas = thetas[table[item].notna()]        
        if np.shape(item_series)[0]>MINIMUM_DATA_POINTS: #probably needs ~50 datapoints but we'll be conservative for now about rejection
            item_series = item_series
            try:
                try:
                    popt,pcov = curve_fit(model,item_thetas,item_series[item],
                                          bounds = bounds, method='trf')
                    #print('size: ', np.shape(item_series))
                    #print('lm:',popt,pcov)
                except:
                    popt,pcov = curve_fit(model,item_thetas,item_series[item],
                                          bounds = bounds, method='dogbox')
                    #print('size: ', np.shape(item_series))
                    #print('dogbox:',popt,pcov)

                if PLOT_ON:
                    import matplotlib.pyplot as plt
                    plt.title('Item '+table.columns[item_num])
                    plt.scatter(item_thetas,item_series[item])

                    plt.plot(np.sort(item_thetas),
                             model(np.sort(item_thetas),*popt))
                    plt.show()
                    print('a :', popt[0], ' b: ', popt[1], ' c: ',popt[2])
                    if FOUR_PL:
                        print('d: ', popt[3])
                    
                all_discriminability[item_num] = popt[0]
                all_difficulty[item_num] = popt[1]
                all_guessing[item_num] = popt[2]
                if FOUR_PL:
                    all_attention_errors[item_num]=popt[3]
            except:
                print('modelling broke, skipping item ', table.columns[item_num]) 
                #this can occur because few responses and they're all correct for instance
    if FOUR_PL:
        return all_discriminability, all_difficulty, all_guessing, all_attention_errors
    else:
        return all_discriminability, all_difficulty, all_guessing


# all_discriminability, all_difficulty, all_guessing = estimate_parameters_for_skill(table)


def plot_item_with_model(thetas, item_num, table, popt, FOUR_PL=True):
    #input student thetas, responses for item
    #thetas: student thetas
    #item_num & table: response table and index to select
    #popt: model parameters
    #plots Item Characterisitic (probability) plot
    item=table.columns[item_num]
    item_series = table[[item]][table[item].notna()]
    item_thetas = thetas[table[item].notna()]
    
    import matplotlib.pyplot as plt
    plt.title('Item '+table.columns[item_num])
    plt.scatter(item_thetas,item_series[item])
    plt.plot(np.sort(item_thetas),
             model(np.sort(item_thetas),*popt))
    plt.show()
    print('a :', popt[0], ' b: ', popt[1], ' c: ',popt[2])
    if FOUR_PL:
        print('d: ', popt[3])    
        
def plot_information_curves(table,thetas,est_params, x_axis=None):
    total_curve=np.zeros(100-1)

    #Information Curves
    theta_range = np.linspace(min(thetas),max(thetas),100)
    for item_num in range(len(table.columns)):
        if ~np.isnan(est_params[0][item_num]):
            a,b,c,d = [i[item_num] for i in est_params]
            plt.plot(theta_range[:-1], np.diff(model(theta_range,a,b,c,d)))
            total_curve+=np.diff(model(theta_range,a,b,c,d))
    plt.title('Info curves for all items')
    if not x_axis==None:
        plt.xlim(x_axis)
    plt.show()

    aa,bb=np.histogram(thetas,100)

    plt.plot(theta_range[:-1],total_curve/sum(total_curve))
    plt.plot(bb[:-1],aa/sum(aa))
    if not x_axis==None:
        plt.xlim(x_axis)
    plt.title('Info curve for entire skill (blue) \n and distribution of theta (orange)')
    plt.show()        
    
    
def distributionsOfEstimatedItemParameters(solvedIRT, FOUR_PL=True):
    plt.hist(solvedIRT.est_params[0], bins = int(2*np.sqrt(len(solvedIRT.est_params[0]))))
    plt.title('discriminability distribution of all items')
    plt.show()


    plt.hist(solvedIRT.est_params[1], bins = int(2*np.sqrt(len(solvedIRT.est_params[1]))))
    plt.title('difficulty distribution of all items')
    plt.show()


    plt.hist(solvedIRT.est_params[2], bins = int(2*np.sqrt(len(solvedIRT.est_params[2]))))
    plt.title('guessing distribution of all items')
    plt.show()

    if FOUR_PL:
        plt.hist(solvedIRT.est_params[3], bins = int(2*np.sqrt(len(solvedIRT.est_params[3]))))
        plt.title('inattention distribution of all items')
        plt.show()

        
def plot_sample_parameter_convergence(solvedIRT, sample_of_items = 10,sample_of_students = 100):
    #plot history of estimates across iterations
    import matplotlib.pyplot as plt
    plt.plot(solvedIRT.discriminability_hx[:,np.random.choice(np.shape(solvedIRT.discriminability_hx)[1],sample_of_items)])
    plt.title('History of estimates of discriminability')
    plt.show()

    plt.plot(solvedIRT.difficulty_hx[:,np.random.choice(np.shape(solvedIRT.difficulty_hx)[1],sample_of_items)])
    plt.title('History of estimates of difficulty')
    plt.show()

    plt.plot(solvedIRT.guessing_hx[:,np.random.choice(np.shape(solvedIRT.guessing_hx)[1],sample_of_items)])
    plt.title('History of estimates of guessing')
    plt.show()

    if True:
        plt.plot(solvedIRT.attention_hx[:,np.random.choice(np.shape(solvedIRT.attention_hx)[1],sample_of_items)])
        plt.title('History of estimates of inattention')
        plt.show()


    plt.plot(solvedIRT.student_thetas_hx[:,np.random.choice(np.shape(solvedIRT.student_thetas_hx)[1],sample_of_students)])
    plt.title('History of estimates of thetas')
    plt.show()

def timeCourseOfParameterConvergence(solvedIRT, exclusion_percentage=5):
    #plot history of estimates across iterations
    #exclude first X% (exclusion_percentage ) of runs b/c they're often large numbers
    import matplotlib.pyplot as plt

    listOfParameters = ['discriminability', 'difficulty', 'guessing', 'attention']
    
    exclude = exclusion_percentage/100
    for param in listOfParameters:
        history = eval('solvedIRT.'+param+'_hx')
        err_history = eval('solvedIRT.'+param+'_error_hx')

        fig, (ax1,ax2,ax3) = plt.subplots(1,3, figsize=(15,5))
        ax1.plot(np.diff(np.nanmean(history,1),1))
        ax1.set_title('Mean history of changes \n in estimate of '+param)

        ax2.plot(np.nanmean(err_history,1))
        ax2.set_title('Mean history of model \n error in estimate of '+param)

        ax3.plot(np.nanmean(err_history[int(np.shape(err_history)[0]*exclude):,:],1))
        ax3.set_title('Mean history of model \n error in estimate of '+param+' \n excluding first '+str(exclusion_percentage)+'% runs')
        plt.show()    

def correlationOfParametersByPerformance(solvedIRT, exclusion_percentage = 10):
    plt.title("student correct % by estimated student ability (theta)")
    plt.scatter(solvedIRT.thetas, solvedIRT.item_correct)
    plt.show()
    print('Correlation: ', np.corrcoef(solvedIRT.thetas, solvedIRT.item_correct)[0][1])

    #X% outlier removal
    outlier=int(len(solvedIRT_5.thetas)*(exclusion_percentage/100))
    indx = np.argsort(solvedIRT_5.thetas)[outlier:-1*outlier]
    plt.title("Student Correct % by \n Estimated Student Ability (theta) \n "+str(exclusion_percentage)+"% OUTLIER removed")
    plt.scatter(solvedIRT.thetas[indx], solvedIRT.item_correct.values[indx])
    plt.show()
    print('Correlation: ', np.corrcoef(solvedIRT.thetas[indx], solvedIRT.item_correct.values[indx])[0][1])
    
    plt.title("item % correct by estimated item difficulty (beta) ")
    plt.scatter(solvedIRT.est_params[1], solvedIRT.student_correct)
    plt.show()
    indx = ~np.isnan(solvedIRT.est_params[1])
    print('Correlation: ', np.corrcoef(solvedIRT.est_params[1][indx], solvedIRT.student_correct[indx])[0][1])

    plt.title("item % correct by estimated item information (alpha) ")
    plt.scatter(solvedIRT.est_params[0], solvedIRT.student_correct)
    plt.show()
    indx = ~np.isnan(solvedIRT.est_params[0])
    print('Correlation: ', np.corrcoef(solvedIRT.est_params[0][indx], solvedIRT.student_correct[indx])[0][1])


def compareRuns(A,B):
    #inputs two solvedIRT objects; outputs correlation between them
    At = A.thetas
    Bt = B.thetas

    plt.hist(A.thetas,bins=100)
    plt.hist(B.thetas,bins=100)
    plt.title('distribution of thetas')
    plt.show()
    print('theta parameter correl: ', np.corrcoef(At[~np.isnan(At)],Bt[~np.isnan(At)])[0][1])

    Ae = A.est_params[0]
    Be = B.est_params[0]
    print('discriminability parameter correl: ', np.corrcoef(Ae[~np.isnan(Ae)],Be[~np.isnan(Ae)])[0][1])

    Ae = A.est_params[1]
    Be = B.est_params[1]
    print('difficulty parameter correl: ',np.corrcoef(Ae[~np.isnan(Ae)],Be[~np.isnan(Ae)])[0][1])

    Ae = A.est_params[2]
    Be = B.est_params[2]
    print('guessing parameter correl: ',np.corrcoef(Ae[~np.isnan(Ae)],Be[~np.isnan(Ae)])[0][1])

    Ae = A.est_params[3]
    Be = B.est_params[3]
    print('atttention parameter correl: ',np.corrcoef(Ae[~np.isnan(Ae)],Be[~np.isnan(Ae)])[0][1])

    
def parallel_estimate_parameters_for_skill(table, thetas, PLOT_ON=True, FOUR_PL=True, est_kernel='trf',
                                          parallel=True, bounds = None):
    #table: table which is all users X all items for that skill, pivoted from pandas
    #needs to include a 'general_score', and a 'num_weights'
    MINIMUM_DATA_POINTS = 30
    #FOUR_PL=False
    from scipy.optimize import curve_fit
    import numpy as np
    
    num_items = len(table.columns)
    
    all_discriminability = np.empty((num_items))
    all_difficulty = np.empty((num_items))
    all_guessing = np.empty((num_items))
    all_errors =  np.empty((num_items))
    all_discriminability[:] = np.nan
    all_difficulty[:] = np.nan
    all_guessing[:] = np.nan
    all_errors =  np.nan
    model = three_param_logistic
    if bounds==None:
        bounds = ((1,-30,0),(100,30,.5)) #fix alpha as positive only
    #p0 = [.5, .5, 0]
    
    if FOUR_PL:
        model = four_param_logistic
        all_attention_errors = np.empty((num_items))
        all_attention_errors[:] = np.nan
        if bounds == None:
            bounds = ((1,-30,0,.5),(100,30,.5,1))
        #p0 = [.5, .5, 0, 1]


    #LATER CAN PARALLELIZE THIS TO ESTIMATE ALL ITEMS AT ONCE
    
#     for item_num in range(num_items):
    def process(item_num):
        item=table.columns[item_num]
        item_series = table[[item]][table[item].notna()]
        item_thetas = thetas[table[item].notna()]        
        if np.shape(item_series)[0]>MINIMUM_DATA_POINTS: #probably needs ~50 datapoints but we'll be conservative for now about rejection
            #if all are correct, or incorrect
            item_series = item_series
            try:
                try:
                    popt,pcov = curve_fit(model,item_thetas,item_series[item],
                                          bounds = bounds, method='trf')
                except:
                    popt,pcov = curve_fit(model,item_thetas,item_series[item],
                                          bounds = bounds, method='dogbox')

                if PLOT_ON:
                    import matplotlib.pyplot as plt
                    plt.title('Item '+table.columns[item_num])
                    plt.scatter(item_thetas,item_series[item])

                    plt.plot(np.sort(item_thetas),
                             model(np.sort(item_thetas),*popt))
                    plt.show()
                    print('a :', popt[0], ' b: ', popt[1], ' c: ',popt[2])
                    if FOUR_PL:
                        print('d: ', popt[3])

                if FOUR_PL:
                    #returns the coefficients, and the ERRORS (est cov matrix)
                    return popt[0], popt[1], popt[2], popt[3], np.sqrt(np.diag(pcov))
        #             return all_discriminability, all_difficulty, all_guessing, all_attention_errors
                else:
                    return popt[0], popt[1], popt[2], np.sqrt(np.diag(pcov))
        #             return all_discriminability, all_difficulty, all_guessing
            except:
                print('modelling broke, skipping item ', table.columns[item_num]) 
        else:
            print('modelling skipped, skipping item ', table.columns[item_num]) 

        #this can occur because few responses and they're all correct for instance
        if FOUR_PL:
            return np.nan, np.nan, np.nan, np.nan, np.asarray([np.nan, np.nan, np.nan, np.nan])
        else:
            return np.nan, np.nan, np.nan, np.asarray([np.nan, np.nan, np.nan])

   
    from joblib import Parallel, delayed
    if parallel:
        results = Parallel(n_jobs=-2)(delayed(process)(i) for i in range(num_items))
    else:
        results = Parallel(n_jobs=1, backend='multiprocessing')(delayed(process)(i) for i in range(num_items))
        
    
    all_discriminability = np.asarray([results[i][0] for i in range(len(results))])
    all_difficulty = np.asarray([results[i][1] for i in range(len(results))])
    all_guessing = np.asarray([results[i][2] for i in range(len(results))])
    
    #stores errors for all parameters (3 or 4, dep on model)
    all_errors = np.asarray([results[i][4] for i in range(len(results))])
    if FOUR_PL:
        all_attention_errors=np.asarray([results[i][3] for i in range(len(results))])
                
    if FOUR_PL:
#             return popt[0], popt[1], popt[2], popt[3]
        return all_discriminability, all_difficulty, all_guessing, all_attention_errors, all_errors
    else:
#             return popt[0], popt[1], popt[2]
        return all_discriminability, all_difficulty, all_guessing, all_errors


# %%capture output


import numpy as np

#this is to estimate probability of student getting item correct
def prob_est(all_thetas, model,all_params):
    return np.asarray([model(all_thetas,*params) for params in all_params]).T

def update_thetas(all_thetas, FOUR_PL, all_est_params, table):
    #thetas, model, item parameters, table of responses
    #inputs: current item parameters (4 if 4PL, etc)
    #table: of all correctness of students x items
    #all probs: all current probabilities for correctness each student and item

    if FOUR_PL:
        model=four_param_logistic
        all_params = zip(all_est_params[0],all_est_params[1],all_est_params[2],all_est_params[3])
    else:
        #set model and current params
        model=three_param_logistic
        all_params = zip(all_est_params[0],all_est_params[1],all_est_params[2])

    all_probs=prob_est(all_thetas,model,all_params) #returns #items by #users
    all_delta_thetas = np.nansum(all_est_params[0]*(table[table.columns]-all_probs),1)/np.nansum(np.power(all_est_params[0],2)*all_probs*(1-all_probs),1)
    
    #MLE denominator
    all_est_delta_confidence = 1/np.nansum(np.power(all_est_params[0],2)*all_probs*(1-all_probs),1)
    return all_delta_thetas, all_est_delta_confidence



def solve_IRT_for_matrix(table, all_thetas = None, iterations = 50, FOUR_PL=True, 
                         show_convergence=10, bounds=None):
    #initialize from scratch
    #history of student ability estimates (& changes over estimation)
    all_student_thetas_hx=np.full((iterations,np.shape(table)[0]), np.nan)
    all_student_delta_thetas_hx=np.full((iterations,np.shape(table)[0]), np.nan)
    #history of item parameter estimates
    all_discriminability_hx=np.full((iterations,len(table.columns)), np.nan)
    all_difficulty_hx=np.full((iterations,len(table.columns)), np.nan)
    all_guessing_hx=np.full((iterations,len(table.columns)), np.nan)
    all_attention_hx=np.full((iterations,len(table.columns)), np.nan)

    #iterations x #items
    all_item_confidence_hx=np.full((iterations,len(table.columns)), np.nan)
    all_item_power_hx=np.full((iterations,len(table.columns)), np.nan)

    #iterations x #students
    all_theta_confidence_hx=np.full((iterations,np.shape(table)[0]), np.nan)
    all_theta_power_hx=np.full((iterations,np.shape(table)[0]), np.nan)

    #capture item paraemter error (iterations x items)
    all_discriminability_error_hx=np.full((iterations,len(table.columns)), np.nan)
    all_difficulty_error_hx=np.full((iterations,len(table.columns)), np.nan)
    all_guessing_error_hx=np.full((iterations,len(table.columns)), np.nan)
    all_attention_error_hx=np.full((iterations,len(table.columns)), np.nan)
    
    #if first run, then assume thetas unknown
    if all_thetas == None:
        all_thetas = table.mean(axis=1) #just create a thetas df
        #replace with random value for initialization
        all_thetas[all_thetas.index] = -.05 + .1*np.random.random((len(all_thetas.index)))


    for iter_num in range(iterations):
        if show_convergence>0:
            print("iteration #", iter_num)

        #if first or last, plot
        if iter_num==0 or iter_num==iterations-1:
            #run estimate function
            #all_discriminability, all_difficulty, all_guessing, all_attention
            #est_params = estimate_parameters_for_skill(table, True,FOUR_PL)
            all_est_params = parallel_estimate_parameters_for_skill(table, all_thetas, 
                                                                    False,FOUR_PL,bounds=bounds)
        else:
            #est_params = estimate_parameters_for_skill(table, False,FOUR_PL)
            all_est_params = parallel_estimate_parameters_for_skill(table, all_thetas, 
                                                                    False,FOUR_PL,bounds=bounds)

        #store history to examine later
        all_discriminability_hx[iter_num,:]=all_est_params[0]
        all_difficulty_hx[iter_num,:]=all_est_params[1]
        all_guessing_hx[iter_num,:] =all_est_params[2]

        #store error history
        all_discriminability_error_hx[iter_num,:]=all_est_params[-1][:,0]
        all_difficulty_error_hx[iter_num,:]=all_est_params[-1][:,1]
        all_guessing_error_hx[iter_num,:] =all_est_params[-1][:,2]

        if FOUR_PL:
            all_attention_hx[iter_num,:]= all_est_params[3]
            #store error history
            all_attention_error_hx[iter_num,:]= all_est_params[-1][:,3]

        all_student_thetas_hx[iter_num,:] = all_thetas

        #update table with new student thetas
        #new theta = old_theta + sum[(a * (response - prob) )]/sum[ a**2 * prob * (1-prob) ]
        # prob = probability of correct response to item i under current ICC model at currently estimated theta (old_theta)
        #delta_thetas = np.sum(all_discriminability*(table[table.columns[:-2]]-all_probs),1)/np.sum(all_discriminability**2*all_probs*(1-all_probs),1)
        #all_delta_thetas = np.nansum(all_est_params[0]*(table[table.columns]-all_probs),1)/np.nansum(np.power(all_est_params[0],2)*all_probs*(1-all_probs),1)

        all_delta_thetas, all_est_delta_confidence = update_thetas(all_thetas, FOUR_PL, all_est_params,table)        
        #update: theta + delta_theta
        all_thetas = np.nansum([all_thetas, all_delta_thetas],0)


        #number of items in estimate for this student's theta
        all_item_power = np.sum(table[table.columns].notna(),1)
        all_student_delta_thetas_hx[iter_num,:] = all_delta_thetas
        all_theta_confidence_hx[iter_num,:] =all_est_delta_confidence
        all_theta_power_hx[iter_num,:] =all_item_power

        #number of students in estimate for this items's params
        all_student_power = np.sum(table[table.columns].notna(),0)

        all_item_power_hx[iter_num,:] =all_student_power

        #if alphas are mostly negative, invert entire theta/beta spectrum
        if sum(all_est_params[0]<0) > sum(all_est_params[0]>=0):
            print('inverting')
            all_thetas*=-1

        if show_convergence > 0:
            show_convergence = int(show_convergence)
            if iter_num%10==0:
                import matplotlib.pyplot as plt
                plt.hist(all_delta_thetas, bins =100)
                plt.show()
                
    #after estimation is complete, calculate SDT accuracy/error
    auc_roc_all = np.full((len(table.columns)), np.nan)
    optimal_threshold_all = np.full((len(table.columns)), np.nan)
    tpr_all = np.full((len(table.columns)), np.nan)
    tnr_all = np.full((len(table.columns)), np.nan)
    
    for item_num in range(len(table.columns)):
        try:
            item=table.columns[item_num]
            item_series = table[[item]][table[item].notna()]
            item_thetas = all_thetas[table[item].notna()]
            model_params = [all_est_params[i][item_num] for i in range(4)]

            #print(np.shape(item_thetas))
            #print(np.shape(model_params), model_params)


            #def estimate_error(performances, thetas, model_params):
            predicted = four_param_logistic(item_thetas,*model_params)
            from sklearn.metrics import roc_curve
            fpri, tpri, thresholds = roc_curve(item_series.values, predicted)
            optimal_thresholdi = thresholds[np.argmax(tpri-fpri)]

            from sklearn.metrics import roc_auc_score
            #return roc auc score and threshold (and tpr and tnr for that threshold)
            auc_roc_all[item_num],optimal_threshold_all[item_num],tpr_all[item_num],tnr_all[item_num] = roc_auc_score(item_series.values, predicted), optimal_thresholdi, tpri[np.argmax(tpri-fpri)], 1-fpri[np.argmax(tpri-fpri)]
            #print('auc_roc,optimal_threshold,tpr,tnr ', auc_roc_all[item_num],optimal_threshold_all[item_num],tpr_all[item_num],tnr_all[item_num])
        except:
            print('skipped AUC ROC, opt thresh, tpr, tnr for item ', table.columns[item_num])
    
    class IRTResults(object):
        question_ids = table.columns
        thetas=all_thetas
        est_params = all_est_params
        delta_thetas=all_delta_thetas
        
        item_power = all_item_power #num items for this theta
        student_power = all_student_power #num students in each item
        student_correct = np.nansum(table,0)/all_student_power #% correct across all students
        item_correct = np.nansum(table,1)/all_item_power #% correct across all items
        est_delta_confidence=all_est_delta_confidence #factors in alpha and prob distribution
        
        delta_thetas_hx = all_student_delta_thetas_hx
        student_thetas_hx = all_student_thetas_hx 
        discriminability_hx = all_discriminability_hx
        difficulty_hx=all_difficulty_hx
        guessing_hx=all_guessing_hx
        attention_hx=all_attention_hx
        item_confidence_hx=all_item_confidence_hx
        item_power_hx=all_item_power_hx

        #iterations x #students
        theta_confidence_hx=all_theta_confidence_hx
        theta_power_hx=all_theta_power_hx

        #capture item paraemter error (iterations x items)
        discriminability_error_hx=all_discriminability_error_hx
        difficulty_error_hx=all_difficulty_error_hx
        guessing_error_hx=all_guessing_error_hx
        attention_error_hx=all_attention_error_hx
            
        auc_roc,optimal_threshold,tpr,tnr = auc_roc_all,optimal_threshold_all,tpr_all,tnr_all
        sample_size = student_power


    return IRTResults

          
            
def submitSnowflakeQuery(query):
    try:
        import snowflake.connector
    except:
        !pip install snowflake-connector-python
        import snowflake.connector


    import base64
    WAREHOUSE = 'COMPUTE_WH'
    USER = 'OMAR_CLAFLIN'
    code = b'T2xsaWUyMSE='
    PASSWORD = base64.b64decode(code).decode("utf-8")
    ACCOUNT='[REDACTED]'
    DATABASE = '[REDACTED]'
    # SCHEMA = ''
    # Gets the version
    ctx = snowflake.connector.connect(
        user=USER,
        password=PASSWORD,
        account=ACCOUNT, 
        database=DATABASE
    #     schema=SCHEMA
        )
    cs = ctx.cursor()
    try:
        cs.execute("USE warehouse COMPUTE_WH;")
        #cs.execute("USE DATABASE [REDACTED];")        
        cs.execute(query)
        all_rows = cs.fetchall()
        print(all_rows)
        #one_row = cs.fetchone()
        #print(one_row[0])
    finally:
        cs.close()
    ctx.close()



def export_object_to_csv(solvedIRT, skill_id, filename='estimatedItemParameters.csv'):
    #CHANGE THIS SO IT UPDATES/APPENDS CSV FILE
    #CHANGE SO IT SHIFTS ALL BETAS TO POSITIVE
    #inputs solved IRT object with all estimated parameters
    #exports a 10 field csv with 4 estimated parameters and 4 error scores for each question_id
    qid = solvedIRT.question_ids
    alpha,beta,gamma,epsilon = [solvedIRT.est_params[i] for i in range(4)]
    #normalize beta
    beta = beta + ((min(beta) - max(beta))/2) - min(beta)
    #could scale beta but then need to scale alpha -- skip for now, just interpretability for laypeople
    alpha_c,beta_c,gamma_c,epsilon_c = [np.asarray(solvedIRT.est_params[-1])[:,i] for i in range(4)]

    auc_roc,optimal_threshold,tpr,tnr,sample_size = solvedIRT.auc_roc,solvedIRT.optimal_threshold,solvedIRT.tpr,solvedIRT.tnr,solvedIRT.sample_size
    export_df = pd.DataFrame({'question_id': qid,
                       'skill_id': skill_id,
                       'discriminability': alpha,
                       'difficulty': beta,
                       'guessing': gamma,
                       'inattention': epsilon,
                       'discriminability_error': alpha_c,
                       'difficulty_error': beta_c,
                       'guessing_error': gamma_c,
                       'inattention_error': epsilon_c,
                       'auc_roc': auc_roc,
                       'optimal_threshold': optimal_threshold,
                       'tpr': tpr,
                       'tnr': tnr,
                       'sample_size': sample_size})
    
    #adding AUC_ROC, optimal_threshold, TPR, TNR, sample_size


    #make directory, store csv in that directory
    import os
#     path = 'itemParametersBySkill'
#     if not os.path.isdir(path):
#         os.mkdir(path)
    if not os.path.isfile(filename):
        export_df.to_csv(filename, index=False)
    else:
        export_df.to_csv(filename, mode='a', header=False, index=False)            


    

In [3]:
try:
    import snowflake.connector
except:
    !pip install snowflake-connector-python
    import snowflake.connector


import base64
WAREHOUSE = 'COMPUTE_WH'
USER = 'OMAR_CLAFLIN'
code = b'T2xsaWUyMSE='
PASSWORD = base64.b64decode(code).decode("utf-8")
ACCOUNT='[REDACTED]'
DATABASE = '[REDACTED]'

    
    
def grabAllDataFromSnowflake(stage='[REDACTED]'):
    #Build stage on snowflake & query for all data

    import time
    starttime = time.time()
    ctx = snowflake.connector.connect(
        user=USER,
        password=PASSWORD,
        account=ACCOUNT, 
        database=DATABASE
    #     schema=SCHEMA
        )
    cs = ctx.cursor()
    # try:
    cs.execute("USE warehouse COMPUTE_WH;")
    cs.execute("USE DATABASE [REDACTED];")
    try:
        cs.execute("DROP STAGE "+stage)
    except:
        print('stage '+ stage+ ' not found.')
        
    cs.execute("CREATE STAGE "+stage)
    # cs.execute("copy into @[REDACTED]/result/data_ from (select * from [REDACTED].views.math_answers_fact) file_format=(compression='gzip');")
    # cs.execute("copy into @[REDACTED]/result/data_ from (select * from [REDACTED].views.math_answers_fact) file_format=(compression='gzip');")
    #This command has a new join to cross-list all skills for each question (that has multiple skill associations) by duplicating rows
    #Also checks to see if that skill-question association is up to date
    cs.execute("copy into @"+stage+"/result/data_ from (select answers.student_id, answers.session_id, answers.created_at, answers.math_question_id, answers.correctness, skill_list.skill_or_subskill_id as RL_TOP_LEVEL_SKILL_ID from [REDACTED].views.math_answers_fact as answers JOIN [REDACTED].prod.math_question_skills as skill_list ON answers.MATH_QUESTION_ID = skill_list.MATH_QUESTION_ID WHERE removed='FALSE' ORDER BY answers.created_at) file_format=(compression='gzip');")

    #datecutoff = '2019-08-09'
    #cs.execute("copy into @"+stage+"/result/data_ from (select answers.student_id, answers.session_id, answers.created_at, answers.math_question_id, answers.correctness, skill_list.skill_or_subskill_id as RL_TOP_LEVEL_SKILL_ID from [REDACTED].views.math_answers_fact as answers JOIN [REDACTED].prod.math_question_skills as skill_list ON answers.MATH_QUESTION_ID = skill_list.MATH_QUESTION_ID WHERE removed='FALSE' AND answers.created_at > "+ datecutoff +" SORT BY answers.created_at) file_format=(compression='gzip');")
    
    print('completed, ', time.time()-starttime, ' seconds.')




    #pull down snowflake stage to local drive
    
    #make local dir
    import os
    if not os.path.exists('AdaptiveMath'):
        os.mkdir('AdaptiveMath')
    else:
        #rename old folder using todays date as archive
        import time
        suffix = str(int(time.time()))
        os.rename('AdaptiveMath','AdaptiveMath'+suffix)
        #make new dir
        os.mkdir('AdaptiveMath')    

    ctx = snowflake.connector.connect(
        user=USER,
        password=PASSWORD,
        account=ACCOUNT, 
        database=DATABASE
    #     schema=SCHEMA
        )
    cs = ctx.cursor()


    import time
    starttime = time.time()
    cs.execute("USE warehouse COMPUTE_WH;")
    cs.execute("USE DATABASE [REDACTED];")
    cs.execute("GET @"+stage +" file:///home/ec2-user/SageMaker/AdaptiveMath/;")
    print('completed, ', time.time()-starttime, ' seconds.')



    #should remove stage from snowflake 

    import time
    starttime = time.time()
    ctx = snowflake.connector.connect(
        user=USER,
        password=PASSWORD,
        account=ACCOUNT, 
        database=DATABASE
    #     schema=SCHEMA
        )
    cs = ctx.cursor()
    cs.execute("USE warehouse COMPUTE_WH;")
    cs.execute("USE DATABASE [REDACTED];")
    cs.execute("drop stage "+stage)


    print('completed, ', time.time()-starttime, ' seconds.')


    #move to s3
    #probably need to clean out s3 location

    import boto3, os
    s3_client = boto3.client('s3')
    bucket = '[REDACTED]'
    for file in os.listdir('AdaptiveMath'):
        if file[-6:]=='csv.gz':
            response=s3_client.upload_file('AdaptiveMath/'+file,bucket,'AdaptiveMathData/'+file)
        else:
            print(file, ' skipped.')
        
        
# grabAllDataFromSnowflake()


In [ ]:
# !pip install --upgrade pip
# !pip install --upgrade --force-reinstall pandas
# !pip install --upgrade --force-reinstall pyarrow
# !pip install --upgrade --force-reinstall snowflake-connector-python
# !pip install --upgrade --force-reinstall sqlalchemy
# !pip install --upgrade --force-reinstall snowflake-sqlalchemy
# !pip install --force-reinstall 'snowflake-connector-python==2.7'  
# !pip install snowflake.connector
# import snowflake.connector



In [5]:
# !pip install pyirt

In [ ]:
#pull from s3 archive to local dir (old files)

import time
starttime=time.time()

from boto3 import client

import os
os.mkdir('AdaptiveMathData')

conn = client('s3')  # again assumes boto.cfg setup, assume AWS S3
for key in conn.list_objects(Bucket='[REDACTED]')['Contents']:
    print(key['Key'])
    conn.download_file('[REDACTED]', key['Key'], key['Key'])

print(time.time()-starttime, ' seconds.')



In [ ]:
import time
starttime=time.time()
# !pip install --upgrade pip
# !pip install 'snowflake-connector-python==2.7' 
list_of_skills = query_for_skills()

print(time.time()-starttime, ' seconds.')

In [ ]:
# i=129
# while np.shape(df)[0]<100:
#     i+=1
#     skill_id = list_of_skills[i]


import time
starttime=time.time()


i=0
skill_id = list_of_skills[i][0]
if True:
#     skill_id='shooraev'

    import time
    start=time.time()
    df = loadAndFilterIntoDataframe(skill_id = skill_id)
    table=returnTable(df)
    print('loading time: ', time.time()-start)
    import numpy as np
    print('# of items: ',len(df), np.shape(df), np.shape(table))


print(i)
print(skill_id)
print(np.shape(df))

print(time.time()-starttime, ' seconds.')




In [ ]:
#solve IRT
import time
starttime = time.time()
solvedIRT = solve_IRT_for_matrix(table, all_thetas = None, iterations = 250, 
                                   FOUR_PL=True, show_convergence=False,
                                   bounds = ((1,-3,0,.5),(100,3,.5,1)))
print('estimating parameter time: ', time.time()-starttime, ' seconds.')

#write out csv to local directory -- change this for lambda or other resource to appropriate dir
#export_object_to_csv(solvedIRT, skill_id, filename)

#print(time.time()-starttime, ' seconds.')


In [ ]:
solvedIRT.est_params

In [ ]:
!pip install pyirt


In [ ]:
#neeeds to be ljist of tuples: user_id,skill_id,boolean correct (1 or 0)
pyirt_df = df.copy()
pyirt_df['correctness']=(pyirt_df['correctness']==100)*1
pyirt_df = list(pyirt_df.itertuples(index=False, name=None))
pyirt_df


In [ ]:

starttime=time.time()
from pyirt import irt

item_param, user_param = irt(pyirt_df)
print(time.time()-starttime, ' seconds')

In [ ]:
item_param

In [ ]:
user_param

In [ ]:
solvedIRT.est_params

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(solvedIRT.est_params[0], [item_param[key]['alpha'] for key in item_param.keys()])
plt.show()

plt.scatter(solvedIRT.est_params[1], [item_param[key]['beta'] for key in item_param.keys()])
plt.show()

plt.scatter(solvedIRT.est_params[2], [item_param[key]['c'] for key in item_param.keys()])
plt.show()

plt.scatter(solvedIRT.thetas, [user_param[key] for key in user_param.keys()])
plt.show()


In [ ]:
def solve_pyIRT_for_matrix(table, all_thetas = None, iterations = 50, FOUR_PL=True, 
                         show_convergence=10, bounds=None):
    #This uses pyIRT instead of my own solver to solve for IRT params
    #initialize from scratch
    #history of student ability estimates (& changes over estimation)
    all_student_thetas_hx=np.full((iterations,np.shape(table)[0]), np.nan)
    all_student_delta_thetas_hx=np.full((iterations,np.shape(table)[0]), np.nan)
    #history of item parameter estimates
    all_discriminability_hx=np.full((iterations,len(table.columns)), np.nan)
    all_difficulty_hx=np.full((iterations,len(table.columns)), np.nan)
    all_guessing_hx=np.full((iterations,len(table.columns)), np.nan)
    all_attention_hx=np.full((iterations,len(table.columns)), np.nan)

    #iterations x #items
    all_item_confidence_hx=np.full((iterations,len(table.columns)), np.nan)
    all_item_power_hx=np.full((iterations,len(table.columns)), np.nan)

    #iterations x #students
    all_theta_confidence_hx=np.full((iterations,np.shape(table)[0]), np.nan)
    all_theta_power_hx=np.full((iterations,np.shape(table)[0]), np.nan)

    #capture item paraemter error (iterations x items)
    all_discriminability_error_hx=np.full((iterations,len(table.columns)), np.nan)
    all_difficulty_error_hx=np.full((iterations,len(table.columns)), np.nan)
    all_guessing_error_hx=np.full((iterations,len(table.columns)), np.nan)
    all_attention_error_hx=np.full((iterations,len(table.columns)), np.nan)
    
    #if first run, then assume thetas unknown
#     if all_thetas == None:
#         all_thetas = table.mean(axis=1) #just create a thetas df
#         #replace with random value for initialization
#         all_thetas[all_thetas.index] = -.05 + .1*np.random.random((len(all_thetas.index)))
#     for iter_num in range(iterations):
#         if show_convergence>0:
#             print("iteration #", iter_num)

#         #if first or last, plot
#         if iter_num==0 or iter_num==iterations-1:
#             #run estimate function
#             #all_discriminability, all_difficulty, all_guessing, all_attention
#             #est_params = estimate_parameters_for_skill(table, True,FOUR_PL)
#             all_est_params = parallel_estimate_parameters_for_skill(table, all_thetas, 
#                                                                     False,FOUR_PL,bounds=bounds)
#         else:
#             #est_params = estimate_parameters_for_skill(table, False,FOUR_PL)
#             all_est_params = parallel_estimate_parameters_for_skill(table, all_thetas, 
#                                                                     False,FOUR_PL,bounds=bounds)

        #store history to examine later
        all_discriminability_hx[iter_num,:]=all_est_params[0]
        all_difficulty_hx[iter_num,:]=all_est_params[1]
        all_guessing_hx[iter_num,:] =all_est_params[2]

        #store error history
        all_discriminability_error_hx[iter_num,:]=all_est_params[-1][:,0]
        all_difficulty_error_hx[iter_num,:]=all_est_params[-1][:,1]
        all_guessing_error_hx[iter_num,:] =all_est_params[-1][:,2]

        if FOUR_PL:
            all_attention_hx[iter_num,:]= all_est_params[3]
            #store error history
            all_attention_error_hx[iter_num,:]= all_est_params[-1][:,3]

        all_student_thetas_hx[iter_num,:] = all_thetas

        #update table with new student thetas
        #new theta = old_theta + sum[(a * (response - prob) )]/sum[ a**2 * prob * (1-prob) ]
        # prob = probability of correct response to item i under current ICC model at currently estimated theta (old_theta)
        #delta_thetas = np.sum(all_discriminability*(table[table.columns[:-2]]-all_probs),1)/np.sum(all_discriminability**2*all_probs*(1-all_probs),1)
        #all_delta_thetas = np.nansum(all_est_params[0]*(table[table.columns]-all_probs),1)/np.nansum(np.power(all_est_params[0],2)*all_probs*(1-all_probs),1)

        all_delta_thetas, all_est_delta_confidence = update_thetas(all_thetas, FOUR_PL, all_est_params,table)        
        #update: theta + delta_theta
        all_thetas = np.nansum([all_thetas, all_delta_thetas],0)


        #number of items in estimate for this student's theta
        all_item_power = np.sum(table[table.columns].notna(),1)
        all_student_delta_thetas_hx[iter_num,:] = all_delta_thetas
        all_theta_confidence_hx[iter_num,:] =all_est_delta_confidence
        all_theta_power_hx[iter_num,:] =all_item_power

        #number of students in estimate for this items's params
        all_student_power = np.sum(table[table.columns].notna(),0)

        all_item_power_hx[iter_num,:] =all_student_power

        #if alphas are mostly negative, invert entire theta/beta spectrum
        if sum(all_est_params[0]<0) > sum(all_est_params[0]>=0):
            print('inverting')
            all_thetas*=-1

        if show_convergence > 0:
            show_convergence = int(show_convergence)
            if iter_num%10==0:
                import matplotlib.pyplot as plt
                plt.hist(all_delta_thetas, bins =100)
                plt.show()
                
    #after estimation is complete, calculate SDT accuracy/error
    auc_roc_all = np.full((len(table.columns)), np.nan)
    optimal_threshold_all = np.full((len(table.columns)), np.nan)
    tpr_all = np.full((len(table.columns)), np.nan)
    tnr_all = np.full((len(table.columns)), np.nan)
    
    for item_num in range(len(table.columns)):
        try:
            item=table.columns[item_num]
            item_series = table[[item]][table[item].notna()]
            item_thetas = all_thetas[table[item].notna()]
            model_params = [all_est_params[i][item_num] for i in range(4)]

            #print(np.shape(item_thetas))
            #print(np.shape(model_params), model_params)


            #def estimate_error(performances, thetas, model_params):
            predicted = four_param_logistic(item_thetas,*model_params)
            from sklearn.metrics import roc_curve
            fpri, tpri, thresholds = roc_curve(item_series.values, predicted)
            optimal_thresholdi = thresholds[np.argmax(tpri-fpri)]

            from sklearn.metrics import roc_auc_score
            #return roc auc score and threshold (and tpr and tnr for that threshold)
            auc_roc_all[item_num],optimal_threshold_all[item_num],tpr_all[item_num],tnr_all[item_num] = roc_auc_score(item_series.values, predicted), optimal_thresholdi, tpri[np.argmax(tpri-fpri)], 1-fpri[np.argmax(tpri-fpri)]
            #print('auc_roc,optimal_threshold,tpr,tnr ', auc_roc_all[item_num],optimal_threshold_all[item_num],tpr_all[item_num],tnr_all[item_num])
        except:
            print('skipped AUC ROC, opt thresh, tpr, tnr for item ', table.columns[item_num])
    
    class IRTResults(object):
        question_ids = table.columns
        thetas=all_thetas
        est_params = all_est_params
        delta_thetas=all_delta_thetas
        
        item_power = all_item_power #num items for this theta
        student_power = all_student_power #num students in each item
        student_correct = np.nansum(table,0)/all_student_power #% correct across all students
        item_correct = np.nansum(table,1)/all_item_power #% correct across all items
        est_delta_confidence=all_est_delta_confidence #factors in alpha and prob distribution
        
        delta_thetas_hx = all_student_delta_thetas_hx
        student_thetas_hx = all_student_thetas_hx 
        discriminability_hx = all_discriminability_hx
        difficulty_hx=all_difficulty_hx
        guessing_hx=all_guessing_hx
        attention_hx=all_attention_hx
        item_confidence_hx=all_item_confidence_hx
        item_power_hx=all_item_power_hx

        #iterations x #students
        theta_confidence_hx=all_theta_confidence_hx
        theta_power_hx=all_theta_power_hx

        #capture item paraemter error (iterations x items)
        discriminability_error_hx=all_discriminability_error_hx
        difficulty_error_hx=all_difficulty_error_hx
        guessing_error_hx=all_guessing_error_hx
        attention_error_hx=all_attention_error_hx
            
        auc_roc,optimal_threshold,tpr,tnr = auc_roc_all,optimal_threshold_all,tpr_all,tnr_all
        sample_size = student_power


    return IRTResults

In [ ]:
#solve IRT using pyIRT library
import time
starttime = time.time()
solvedIRT_py = solve_pyIRT_for_matrix(table, all_thetas = None, iterations = 250, 
                                   FOUR_PL=True, show_convergence=True,
                                   bounds = ((1,-3,0,.5),(100,3,.5,1)))
print('estimating parameter time: ', time.time()-starttime, ' seconds.')

#write out csv to local directory -- change this for lambda or other resource to appropriate dir
export_object_to_csv(solvedIRT_py, skill_id, filename)

print(time.time()-starttime, ' seconds.')




In [ ]:
#solve IRT
import time
starttime = time.time()
solvedIRT_a = solve_IRT_for_matrix(table, all_thetas = None, iterations = 250, 
                                   FOUR_PL=True, show_convergence=True,
                                   bounds = ((1,-3,0,.5),(100,3,.5,1)))
print('estimating parameter time: ', time.time()-starttime, ' seconds.')

#write out csv to local directory -- change this for lambda or other resource to appropriate dir
export_object_to_csv(solvedIRT_a, skill_id, filename)

print(time.time()-starttime, ' seconds.')



In [ ]:
df

In [ ]:
# solvedIRT_7 = solve_IRT_for_matrix(table, all_thetas = None, iterations = 1, 
#                                            FOUR_PL=True, show_convergence=0,
#                                            bounds = ((1,-3,0,.5),(100,3,.5,1)))



In [ ]:
# solvedIRT_8 = solve_IRT_for_matrix(table, all_thetas = None, iterations = 50, 
#                                            FOUR_PL=True, show_convergence=1,
#                                            bounds = ((1,-3,0,.5),(100,3,.5,1)))


In [ ]:

# todaydate = str(datetime.date.today())
# todaydate

In [ ]:
#downloadFromS3()
import datetime
todaydate = str(datetime.date.today())
todaydate = todaydate[:4]+todaydate[5:7]+todaydate[8:]

tablename = "PYIRTItemParameterEstimates"+todaydate
filename= tablename+'.csv'

#get all skills
list_of_skills = query_for_skills()
los = [i[0] for i in list_of_skills]
print('fetched ', str(len(los)), ' skills.')

count=0
for skill_id in los:
    print('skill_id: ', skill_id)
    
    try:
        #ETL for this skill
        import time
        start=time.time()
        df = loadAndFilterIntoDataframe(skill_id = skill_id)
        table=returnTable(df)
        print('loading time: ', time.time()-start)
        import numpy as np
        print('# of items: ',len(df), np.shape(df), np.shape(table))


        #solve IRT
        starttime = time.time()
        solvedIRT_7 = solve_IRT_for_matrix(table, all_thetas = None, iterations = 250, 
                                           FOUR_PL=True, show_convergence=0,
                                           bounds = ((1,-3,0,.5),(100,3,.5,1)))
        print('estimating parameter time: ', time.time()-starttime, ' seconds.')

        #write out csv to local directory -- change this for lambda or other resource to appropriate dir
        export_object_to_csv(solvedIRT_7, skill_id, filename)
    except:
        print('skipping ',skill_id, ' either loading or estimating broke.')

        
    if (count/200) == int(count/200) or count ==len(skill_id)-1:
        print('exporting to snowflake...')
        import boto3, os
        s3_client = boto3.client('s3')
        response=s3_client.upload_file(filename,'product-snowflake-testing',filename)

        #create snowflake table
        submitSnowflakeQuery("create or replace table product_testing."+tablename+" (question_id TEXT, "+
                             "skill_id TEXT, discriminability FLOAT, difficulty FLOAT, guessing FLOAT, inattention FLOAT, "+
                            "discriminability_error FLOAT, difficulty_error FLOAT, guessing_error FLOAT, "+
                             "inattention_error FLOAT, auc_roc FLOAT, optimal_threshold FLOAT, tpr FLOAT, tnr FLOAT, "+
                            "sample_size INTEGER)")

        #adding AUC_ROC, optimal_threshold, TPR, TNR, sample_size



        #copy data into sf table
        submitSnowflakeQuery("copy into product_testing."+tablename+" from @public.product_testing_stage/"+filename)

    count+=1

# #test with query -- not necessary for production
# submitSnowflakeQuery("select * from product_testing.itemParameters")

In [ ]:
#upload to s3

import logging
import boto3
from botocore.exceptions import ClientError
import os


def upload_file(file_name, bucket, object_name=None):
    """Upload a file to an S3 bucket

    :param file_name: File to upload
    :param bucket: Bucket to upload to
    :param object_name: S3 object name. If not specified then file_name is used
    :return: True if file was uploaded, else False
    """

    # If S3 object_name was not specified, use file_name
    if object_name is None:
        object_name = os.path.basename(file_name)

    # Upload the file
    s3_client = boto3.client('s3')
    try:
        response = s3_client.upload_file(file_name, bucket, object_name)
    except ClientError as e:
        logging.error(e)
        return False
    return True

tablename='ItemParameterEstimates20221106'
bucket_name ='data-science-[REDACTED]'
file_name = tablename+'.csv'

s3 = boto3.client('s3')
with open(file_name, "rb") as f:
    s3.upload_fileobj(f, bucket_name, file_name)

In [ ]:
# tablename = "ItemParameterEstimates09092022"
# filename='estimatedItemParameters09092022.csv'


# import pandas as pd
# t=pd.read_csv(filename)
# # los = t['skill_id'].unique()
# # len(los), 2408